# Patent Landscaping — Snorkel vs MAS across 5 Bergeaud domains

The **same** pipeline as the self-driving experiment (`colab_experiment.ipynb`), now run for the
other 5 Bergeaud & Verluise technologies, each with its OWN gold set (`DataSet/학습용 음성/training_<domain>.csv`):

- **additivemanufacturing, blockchain, computervision, genomeediting, hydrogenstorage**

Per domain, the candidate pool was collected from the PatentsView bulk with that domain's official
CPC ∩ keyword query (S2 Appendix) — committed as `training_expanded_clean.csv`. MAS already labeled
each `candidate_all` locally (committed `mas_ranked_scores.csv`); Snorkel labels on Colab here.
Each domain trains 4 models: `{snorkel,mas} × {noood (--ood-n 0), ood (all OOD)}`.

Runtime → GPU (T4/L4) → Run all. Models persist to Drive (restore on a fresh session, no retrain).

In [ ]:
# 1) setup
import os
%cd /content
REPO = 'https://github.com/PassionChicken-Leesuin/Patent_Landscaping_Final.git'
if not os.path.exists('/content/Patent_Landscaping_Final'):
    !git clone $REPO
%cd /content/Patent_Landscaping_Final
!git pull
!pip -q install snorkel transformers datasets scikit-learn accelerate

In [ ]:
# 2) mount Drive (persistent model store)
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/patent_landscaping_outputs'
os.makedirs(DRIVE, exist_ok=True); os.makedirs('outputs', exist_ok=True)
print('Drive store:', DRIVE)

In [ ]:
# 3) config
DOMAINS = ['additivemanufacturing', 'blockchain', 'computervision', 'genomeediting', 'hydrogenstorage']
EPOCHS, MAX_LEN = 4, 256
CONFIGS = [('snorkel', '--ood-n 0', 'noood'), ('mas', '--ood-n 0', 'noood'),
           ('snorkel', '', 'ood'),          ('mas', '', 'ood')]

In [ ]:
# 4) data prep per domain: gold->eval + OOD pool, candidate_all (pool+OOD), Snorkel labels.
#    MAS labels are already committed (DataSet/mas/<domain>/mas_ranked_scores.csv).
for d in DOMAINS:
    print(f'\n========== {d} ==========')
    !python -m scripts.build_dataset --domain $d
    !python -m scripts.build_candidate_all --domain $d
    !python -m scripts.run_snorkel --domain $d

In [ ]:
# 5) train each (domain x arm x {noood,ood}). Restore from Drive if present.
import shutil
FORCE = False

def ensure(name, run_args):
    local, dstore = f'outputs/scibert_{name}', f'{DRIVE}/scibert_{name}'
    if not FORCE and os.path.isfile(f'{local}/config.json'):
        if not os.path.isfile(f'{dstore}/config.json'):
            shutil.copytree(local, dstore, dirs_exist_ok=True)
            if os.path.isfile(f'outputs/metrics_{name}.json'): shutil.copy(f'outputs/metrics_{name}.json', DRIVE)
        print(f'[{name}] local'); return
    if not FORCE and os.path.isfile(f'{dstore}/config.json'):
        shutil.copytree(dstore, local, dirs_exist_ok=True)
        if os.path.isfile(f'{DRIVE}/metrics_{name}.json'): shutil.copy(f'{DRIVE}/metrics_{name}.json', 'outputs/')
        print(f'[{name}] restored'); return
    print(f'[{name}] training ...')
    get_ipython().system(f'python -m scripts.run_downstream {run_args} --epochs {EPOCHS} --max-len {MAX_LEN}')
    if os.path.isfile(f'{local}/config.json'):
        shutil.copytree(local, dstore, dirs_exist_ok=True)
        if os.path.isfile(f'outputs/metrics_{name}.json'): shutil.copy(f'outputs/metrics_{name}.json', DRIVE)
        print(f'[{name}] done + saved')

for d in DOMAINS:
    for arm, ood_flag, tag in CONFIGS:
        name = f'{d}_{arm}_{tag}'
        ensure(name, f'--domain {d} --arm {arm} --unified {ood_flag} --tag {tag}')

In [ ]:
# 6) train/val curves
from src.downstream.plots import plot_history
for d in DOMAINS:
    for arm, _, tag in CONFIGS:
        n = f'{d}_{arm}_{tag}'
        try: plot_history(n)
        except Exception: print(f'[{n}] no history')

In [ ]:
# 7) compare @0.5 across all domains. acc=(TP+TN)/all. auc_seed_vs_hard = SEED vs ANTISEED-manual.
import json
def row(name):
    p = f'outputs/metrics_{name}.json'
    if not os.path.isfile(p): return f'{name:34s} (missing)'
    r = json.load(open(p))
    seed = r['by_expansion_level'].get('SEED', {}).get('recall(TP rate)', float('nan'))
    return (f"{name:34s} acc={r['accuracy']:.3f} F1={r['macro_f1']:.3f} AUC={r['auc']:.3f} "
            f"AUC(vs hard)={r.get('auc_seed_vs_hard', float('nan')):.3f} "
            f"R={r['recall']:.3f} P={r['precision']:.3f} SEEDrec={seed:.3f}")
for d in DOMAINS:
    print(f'\n--- {d} ---')
    for arm, _, tag in CONFIGS:
        print(row(f'{d}_{arm}_{tag}'))

## 8) Threshold calibration + sweep per domain (no retrain)
Tunes the threshold on validation (never gold) and shows the precision/recall tradeoff.

In [ ]:
for d in DOMAINS:
    print(f'\n========== {d} ==========')
    !python -m scripts.calibrate_eval --domain $d
    !python -m scripts.threshold_sweep --domain $d